# 1/3 — Dataset prep + EDA (Kaggle, no GPU needed for this notebook)

Companion to `docs/FINE_TUNING_YOLO_GUIDE.md`. This is step 1 of 3 — deliberately split from training (`02_train.ipynb`) so you get a reviewable class-distribution report *before* spending any GPU time, and so a training-session cutoff never means re-downloading LVIS (~20.7GB) from scratch.

**What this notebook produces** (as this notebook's committed Output, which `02_train.ipynb` attaches as an input dataset — see the last cell):
- `lostfound/` — the merged, remapped 48-class training set
- `coco_forgetting_check/` — the automatic catastrophic-forgetting eval set
- `eda_report/` — per-class instance counts, an imbalance flag, and a chart

Run top to bottom. GPU accelerator is NOT required for this notebook (CPU is fine — nothing here trains a model).

In [ ]:
!pip install -q ultralytics==8.4.106
import ultralytics
ultralytics.checks()

## 1. Force Ultralytics to download + convert COCO, then LVIS

**COCO first, LVIS second — not the other way around, and not independently.** LVIS reuses COCO's own `train2017`/`val2017` JPEGs, but Ultralytics' `lvis.yaml` doesn't know that — its own `download:` block fetches `train2017.zip`(19G)+`val2017.zip`(1G) directly into `datasets/lvis/images/`, a **second, separate copy** of the exact same ~20GB of images `coco.yaml` already downloads into `datasets/coco/images/`. Triggering both independently (the old order) silently doubles the download and is why this step can blow past Kaggle's working-disk limit even though the final dataset is nowhere near that big. The next three cells: download COCO's images once, symlink LVIS's expected image directory to that same copy, then trigger LVIS — `safe_download()` only re-fetches a zip if it's missing at that exact path, so through the symlink it finds COCO's zip already there and skips straight to LVIS's own much smaller `lvis-labels-segments.zip`.

In [ ]:
from ultralytics import YOLO, settings
print("datasets_dir:", settings["datasets_dir"])

model = YOLO("yolo26n.pt")
model.train(data="coco.yaml", epochs=1, imgsz=640)  # triggers COCO download+convert (train2017 + val2017); interrupt after epoch 1 starts

In [ ]:
import os
from pathlib import Path

datasets_dir = Path(settings["datasets_dir"])
lvis_images = datasets_dir / "lvis" / "images"
lvis_images.parent.mkdir(parents=True, exist_ok=True)
if not lvis_images.exists():
    os.symlink(datasets_dir / "coco" / "images", lvis_images)
print("lvis/images ->", lvis_images.resolve() if lvis_images.is_symlink() else "already a real directory (not symlinked, left as-is)")

In [ ]:
model = YOLO("yolo26n.pt")
model.train(data="lvis.yaml", epochs=1, imgsz=640)  # with images/ symlinked above, this should only fetch lvis-labels-segments.zip, not re-download train2017/val2017

## 2. Build the merged 48-class training dataset

Upload `finetuning/prepare_dataset.py`, `finetuning/prepare_coco_forgetting_check.py`, and `finetuning/eda.py` from this repo as a Kaggle Dataset (or **Add Data → Upload**) and attach it to this notebook before running the next cell. Keeps 20 of COCO's 80 classes and adds 28 new LVIS-backed classes — see `prepare_dataset.py`'s module docstring or `docs/FINE_TUNING_YOLO_GUIDE.md` Section 2 for the full rationale. **Add Data** attaches an uploaded dataset read-only under `/kaggle/input/<slug>/` — it does **not** land in `/kaggle/working/` (this notebook's cwd), so the next cell copies the three scripts into place before anything tries to run them.

In [ ]:
import shutil, glob

SCRIPTS_INPUT_DIR = "/kaggle/input/<scripts-slug>"  # <-- EDIT: the dataset/upload containing prepare_dataset.py, prepare_coco_forgetting_check.py, eda.py

# Add Data attaches datasets read-only under /kaggle/input/ — never into /kaggle/working/,
# which is this notebook's cwd for the !python cells below. Copy the scripts into place
# first, or every !python call below fails with "No such file or directory".
for f in glob.glob(f"{SCRIPTS_INPUT_DIR}/*.py"):
    shutil.copy(f, ".")
copied = glob.glob("*.py")
assert {"prepare_dataset.py", "prepare_coco_forgetting_check.py", "eda.py"} <= set(copied), f"Missing expected script(s) — only found {copied}. Check SCRIPTS_INPUT_DIR above."
print("Copied:", copied)

In [ ]:
DATASETS_DIR = settings["datasets_dir"]  # printed by the "trigger COCO download" cell above

!python prepare_dataset.py \
    --lvis-root {DATASETS_DIR}/lvis \
    --coco-root {DATASETS_DIR}/coco \
    --output-dir /kaggle/working/lostfound \
    --coco-sample-size 750

## 3. Build the forgetting-check set (fully automatic)

Unlike the golden set (which needs real self-captured photos, done separately — see `finetuning/README.md`), this one needs no new annotation: it's a filtered/remapped sample of COCO's own `val2017` ground truth.

In [ ]:
!python prepare_coco_forgetting_check.py \
    --coco-root {DATASETS_DIR}/coco \
    --output-dir /kaggle/working/coco_forgetting_check \
    --sample-size 75

## 4. EDA — per-class instance counts and imbalance check

**Read this before moving on.** `prepare_dataset.py` only ever printed how many *images* matched each source — never how many box *instances* per class actually survived the merge. A class can show up in plenty of images but still have very few boxes. `eda.py` counts real per-class instances (train+val), flags anything under the imbalance floor (default 50), and writes a CSV + chart you can actually look at.

In [ ]:
!python eda.py --dataset-dir /kaggle/working/lostfound --output-dir /kaggle/working/eda_report --imbalance-floor 50

In [ ]:
from IPython.display import Image, display
import json

display(Image(filename="/kaggle/working/eda_report/class_distribution.png"))
print(json.dumps(json.load(open("/kaggle/working/eda_report/eda_summary.json")), indent=2))

## 5. Review gate — decide before training, not after

Look at `eda_report/class_distribution.csv` and the chart above:
- **Zero-instance classes** mean training will never learn them at all — fix the dataset before training, don't discover this from a confusing eval result later.
- **Classes under the imbalance floor** are candidates for a Roboflow Universe fallback (`docs/FINE_TUNING_YOLO_GUIDE.md` Section 3.2) — check per-class, since LVIS coverage varies a lot by class.
- A large `imbalance_ratio_max_over_min` is expected (LVIS classes aren't uniformly common) — the question is whether any *individual* class is too sparse to learn anything, not whether the distribution is perfectly even.

## 6. Persist this notebook's output for 02_train.ipynb

**File → Save Version** now (even without re-running) — this commits `/kaggle/working` (i.e. `lostfound/`, `coco_forgetting_check/`, `eda_report/`) as this notebook's Output, which becomes a Kaggle Dataset. In `02_train.ipynb`, **Add Data** → this notebook's output version, instead of re-downloading or re-merging anything there.